# Gold ICU Daily Fact

Build the ICU daily clinical monitoring fact.

Output:
- `fact_icu_daily_monitoring`

Grain:
- one row per `stay_id` + `clinical_date`

The fact combines ICU stay context with daily:
- vital signs
- laboratory results
- fluid input/output
- procedures

Cleaning and validity flags already created in Silver are reused rather than recalculated in Gold.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_SCHEMA = "cdr_silver.processed_mimic_iv"
GOLD_SCHEMA = "cdr_gold.analytics_mimic_iv"

print(f"Silver source: {SILVER_SCHEMA}")
print(f"Gold target:   {GOLD_SCHEMA}")

In [0]:
# ICU clinical sources

icustays = spark.table(f"{SILVER_SCHEMA}.icustays")
chartevents = spark.table(f"{SILVER_SCHEMA}.chartevents")
labevents = spark.table(f"{SILVER_SCHEMA}.labevents")
inputevents = spark.table(f"{SILVER_SCHEMA}.inputevents")
outputevents = spark.table(f"{SILVER_SCHEMA}.outputevents")
procedureevents = spark.table(f"{SILVER_SCHEMA}.procedureevents")

# Shared Gold dimensions
dim_patient = spark.table(f"{GOLD_SCHEMA}.dim_patient")
dim_date = spark.table(f"{GOLD_SCHEMA}.dim_date")
dim_careunit = spark.table(f"{GOLD_SCHEMA}.dim_careunit")

print("ICU daily fact source tables loaded.")

In [0]:
# Build ICU stay-day spine
# Grain: one row per stay_id + clinical_date

icu_daily_spine = (
    icustays
    .filter(
        F.col("stay_id").isNotNull()
        & F.col("intime").isNotNull()
    )
    .withColumn(
        "_end_date",
        F.coalesce(
            F.to_date("outtime"),
            F.to_date("intime")
        )
    )
    .withColumn(
        "clinical_date",
        F.explode(
            F.sequence(
                F.to_date("intime"),
                F.col("_end_date"),
                F.expr("INTERVAL 1 DAY")
            )
        )
    )
    .withColumn(
        "icu_day_number",
        F.datediff(
            F.col("clinical_date"),
            F.to_date("intime")
        ) + F.lit(1)
    )
    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        "clinical_date",
        "icu_day_number",
        "intime",
        "outtime",
        F.col("los").alias("icu_los_days"),
        "icu_stay_complete_flag",
        "los_availability_flag",
        "first_careunit",
        "last_careunit"
    )
)

display(
    icu_daily_spine.agg(
        F.count("*").alias("stay_days"),
        F.countDistinct("stay_id").alias("unique_stays"),
        F.countDistinct(
            F.struct("stay_id", "clinical_date")
        ).alias("unique_stay_days")
    )
)

In [0]:
# Daily vital-sign summary
# Silver already handles numeric validity, plausibility and temperature conversion.

VITAL_ITEM_IDS = [
    220045,  # Heart rate
    220210,  # Respiratory rate
    220277,  # SpO2
    223761,  # Temperature F
    223762   # Temperature C
]

vital_events = (
    chartevents
    .filter(
        F.col("stay_id").isNotNull()
        & F.col("charttime").isNotNull()
        & F.col("itemid").isin(VITAL_ITEM_IDS)
        & (F.col("numeric_value_valid_flag") == 1)
    )
    .withColumn(
        "_vital_name",
        F.when(F.col("itemid") == 220045, "heart_rate")
         .when(F.col("itemid") == 220210, "respiratory_rate")
         .when(F.col("itemid") == 220277, "spo2")
         .when(
             F.col("itemid").isin(223761, 223762),
             "temperature"
         )
    )
    .withColumn(
        "_vital_value",
        F.when(
            F.col("itemid").isin(223761, 223762),
            F.col("temperature_c")
        ).otherwise(
            F.col("valuenum")
        )
    )
    .filter(
        (
            F.col("itemid").isin(223761, 223762)
            & (F.col("temperature_plausible_flag") == 1)
        )
        |
        (
            ~F.col("itemid").isin(223761, 223762)
            & (F.col("vital_implausible_flag") == 0)
        )
    )
    .withColumn(
        "clinical_date",
        F.to_date("charttime")
    )
)

vital_daily = (
    vital_events
    .groupBy(
        "stay_id",
        "clinical_date"
    )
    .agg(
        F.count("*").alias("vital_observation_count"),

        F.countDistinct("_vital_name")
        .alias("distinct_vital_measurements"),

        F.sum(
            F.when(F.col("warning") == 1, 1).otherwise(0)
        ).alias("vital_warning_count"),

        F.avg(
            F.when(
                F.col("_vital_name") == "heart_rate",
                F.col("_vital_value")
            )
        ).alias("heart_rate_avg"),

        F.min(
            F.when(
                F.col("_vital_name") == "heart_rate",
                F.col("_vital_value")
            )
        ).alias("heart_rate_min"),

        F.max(
            F.when(
                F.col("_vital_name") == "heart_rate",
                F.col("_vital_value")
            )
        ).alias("heart_rate_max"),

        F.avg(
            F.when(
                F.col("_vital_name") == "respiratory_rate",
                F.col("_vital_value")
            )
        ).alias("respiratory_rate_avg"),

        F.min(
            F.when(
                F.col("_vital_name") == "respiratory_rate",
                F.col("_vital_value")
            )
        ).alias("respiratory_rate_min"),

        F.max(
            F.when(
                F.col("_vital_name") == "respiratory_rate",
                F.col("_vital_value")
            )
        ).alias("respiratory_rate_max"),

        F.avg(
            F.when(
                F.col("_vital_name") == "spo2",
                F.col("_vital_value")
            )
        ).alias("spo2_avg"),

        F.min(
            F.when(
                F.col("_vital_name") == "spo2",
                F.col("_vital_value")
            )
        ).alias("spo2_min"),

        F.max(
            F.when(
                F.col("_vital_name") == "spo2",
                F.col("_vital_value")
            )
        ).alias("spo2_max"),

        F.avg(
            F.when(
                F.col("_vital_name") == "temperature",
                F.col("_vital_value")
            )
        ).alias("temperature_c_avg"),

        F.min(
            F.when(
                F.col("_vital_name") == "temperature",
                F.col("_vital_value")
            )
        ).alias("temperature_c_min"),

        F.max(
            F.when(
                F.col("_vital_name") == "temperature",
                F.col("_vital_value")
            )
        ).alias("temperature_c_max")
    )
)

display(
    vital_daily.agg(
        F.count("*").alias("vital_stay_days"),
        F.sum("vital_observation_count").alias("vital_observations")
    )
)

In [0]:
# Daily laboratory summary
# Labs may not contain hadm_id, so map by patient, optional admission,
# and ICU timestamp window.

LAB_ITEM_IDS = [
    50912, 52546,        # Creatinine
    51222,               # Hemoglobin
    51300, 51301,        # WBC
    50983, 52623,        # Sodium
    50971, 52610,        # Potassium
    50813, 52442, 53154  # Lactate
]

lab_candidates = (
    labevents
    .filter(
        F.col("charttime").isNotNull()
        & F.col("itemid").isin(LAB_ITEM_IDS)
        & (F.col("numeric_value_valid_flag") == 1)
    )
    .filter(
        (
            ~F.col("itemid").isin(50813, 52442, 53154)
        )
        |
        (
            F.col("itemid").isin(50813, 52442, 53154)
            & (F.col("lactate_implausible_flag") == 0)
        )
    )
    .withColumn(
        "_lab_name",
        F.when(
            F.col("itemid").isin(50912, 52546),
            "creatinine"
        )
        .when(F.col("itemid") == 51222, "hemoglobin")
        .when(F.col("itemid").isin(51300, 51301), "wbc")
        .when(F.col("itemid").isin(50983, 52623), "sodium")
        .when(F.col("itemid").isin(50971, 52610), "potassium")
        .when(
            F.col("itemid").isin(50813, 52442, 53154),
            "lactate"
        )
    )
)

icu_lab_windows = (
    icustays
    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        "intime",
        "outtime"
    )
)

lab_icu_events = (
    lab_candidates.alias("l")
    .join(
        F.broadcast(icu_lab_windows.alias("i")),
        (
            (F.col("l.subject_id") == F.col("i.subject_id"))
            &
            (
                F.col("l.hadm_id").isNull()
                | (F.col("l.hadm_id") == F.col("i.hadm_id"))
            )
            &
            (
                (
                    F.col("i.outtime").isNotNull()
                    & F.col("l.charttime").between(
                        F.col("i.intime"),
                        F.col("i.outtime")
                    )
                )
                |
                (
                    F.col("i.outtime").isNull()
                    & (F.col("l.charttime") >= F.col("i.intime"))
                    & (
                        F.to_date("l.charttime")
                        == F.to_date("i.intime")
                    )
                )
            )
        ),
        "inner"
    )
    .select(
        F.col("i.stay_id").alias("stay_id"),
        F.to_date(F.col("l.charttime")).alias("clinical_date"),
        F.col("l._lab_name").alias("_lab_name"),
        F.col("l.valuenum").alias("valuenum"),
        F.col("l.abnormal_lab_flag").alias("abnormal_lab_flag")
    )
)

lab_daily = (
    lab_icu_events
    .groupBy(
        "stay_id",
        "clinical_date"
    )
    .agg(
        F.count("*").alias("lab_result_count"),

        F.countDistinct("_lab_name")
        .alias("distinct_lab_tests"),

        F.sum(
            F.when(
                F.col("abnormal_lab_flag") == 1,
                1
            ).otherwise(0)
        ).alias("abnormal_lab_count"),

        F.round(
            F.avg(
                F.when(
                    F.col("_lab_name") == "creatinine",
                    F.col("valuenum")
                )
            ),
            2
        ).alias("creatinine_avg"),

        F.round(
            F.avg(
                F.when(
                    F.col("_lab_name") == "hemoglobin",
                    F.col("valuenum")
                )
            ),
            2
        ).alias("hemoglobin_avg"),

        F.round(
            F.avg(
                F.when(
                    F.col("_lab_name") == "wbc",
                    F.col("valuenum")
                )
            ),
            2
        ).alias("wbc_avg"),

        F.round(
            F.avg(
                F.when(
                    F.col("_lab_name") == "sodium",
                    F.col("valuenum")
                )
            ),
            2
        ).alias("sodium_avg"),

        F.round(
            F.avg(
                F.when(
                    F.col("_lab_name") == "potassium",
                    F.col("valuenum")
                )
            ),
            2
        ).alias("potassium_avg"),

        F.round(
            F.avg(
                F.when(
                    F.col("_lab_name") == "lactate",
                    F.col("valuenum")
                )
            ),
            2
        ).alias("lactate_avg")
    )
)

display(
    lab_daily.agg(
        F.count("*").alias("lab_stay_days"),
        F.sum("lab_result_count").alias("lab_results"),
        F.sum("abnormal_lab_count").alias("abnormal_lab_results")
    )
)

In [0]:
# Daily retained fluid input
# Silver supplies standardised mL and quality flags.
# Multi-day inputs are allocated proportionally across calendar days.

NON_ICU_INPUT_IDS = [
    226364,  # OR Crystalloid Intake
    226367,  # OR FFP Intake
    226368,  # OR Packed RBC Intake
    226377   # PACU PO Intake
]

clean_input_events = (
    inputevents
    .filter(
        F.col("stay_id").isNotNull()
        & F.col("amount_ml").isNotNull()
        & (F.col("input_amount_valid_flag") == 1)
        & (F.col("input_duration_valid_flag") == 1)
        & (F.col("input_amount_extreme_flag") == 0)
    )

    # Exclude pre-admission / non-ICU summaries
    .filter(
        F.col("ordercategoryname").isNull()
        | (F.col("ordercategoryname") != "16-Pre Admission/Non-ICU")
    )

    # Exclude known OR / PACU summary records
    .filter(
        ~F.col("itemid").isin(NON_ICU_INPUT_IDS)
    )
)

input_daily = (
    clean_input_events

    # One row per calendar day touched by the event
    .withColumn(
        "clinical_date",
        F.explode(
            F.sequence(
                F.to_date("starttime"),
                F.to_date("endtime")
            )
        )
    )

    .withColumn(
        "_day_start",
        F.col("clinical_date").cast("timestamp")
    )
    .withColumn(
        "_day_end",
        F.date_add(
            F.col("clinical_date"),
            1
        ).cast("timestamp")
    )

    # Portion of the event inside each calendar day
    .withColumn(
        "_overlap_start",
        F.greatest(
            F.col("starttime"),
            F.col("_day_start")
        )
    )
    .withColumn(
        "_overlap_end",
        F.least(
            F.col("endtime"),
            F.col("_day_end")
        )
    )
    .withColumn(
        "_overlap_seconds",
        F.col("_overlap_end").cast("long")
        - F.col("_overlap_start").cast("long")
    )
    .filter(
        F.col("_overlap_seconds") > 0
    )

    # Allocate total event volume proportionally
    .withColumn(
        "_allocated_input_ml",
        F.col("amount_ml")
        * (
            F.col("_overlap_seconds")
            /
            (
                F.col("endtime").cast("long")
                - F.col("starttime").cast("long")
            )
        )
    )

    .groupBy(
        "stay_id",
        "clinical_date"
    )
    .agg(
        F.count("*").alias("input_event_count"),

        F.round(
            F.sum("_allocated_input_ml"),
            2
        ).alias("total_input_ml")
    )
)

display(
    input_daily.agg(
        F.count("*").alias("input_stay_days"),
        F.countDistinct("stay_id").alias("stays_with_input"),
        F.sum("total_input_ml").alias("allocated_input_ml")
    )
)

In [0]:
# Daily retained fluid output
# Silver handles negative/extreme value flags.
# Gold removes known non-ICU and irrigation summary records.

EXCLUDED_OUTPUT_IDS = [
    226632,  # Cath Lab
    226626,  # OR EBL
    226627,  # OR Urine
    226628,  # PACU Drains
    226629,  # PACU EBL
    226630,  # PACU Gastric
    226631,  # PACU Urine
    226633,  # Pre-Admission

    227487,  # GU Irrigant Type
    227488,  # GU Irrigant Volume In
    227489,  # GU Irrigant/Urine Volume Out
    226566   # Urine and GU Irrigant Out
]

output_daily = (
    outputevents
    .filter(
        F.col("stay_id").isNotNull()
        & F.col("charttime").isNotNull()
        & F.col("value").isNotNull()
        & (F.col("output_value_valid_flag") == 1)
        & (F.col("output_value_extreme_flag") == 0)
        & (~F.col("itemid").isin(EXCLUDED_OUTPUT_IDS))
    )
    .withColumn(
        "clinical_date",
        F.to_date("charttime")
    )
    .groupBy(
        "stay_id",
        "clinical_date"
    )
    .agg(
        F.count("*").alias("output_event_count"),

        F.round(
            F.sum("value"),
            2
        ).alias("total_output_ml")
    )
)

display(
    output_daily.agg(
        F.count("*").alias("output_stay_days"),
        F.countDistinct("stay_id").alias("stays_with_output"),
        F.sum("total_output_ml").alias("total_output_ml")
    )
)

In [0]:
# Daily ICU procedure summary

procedure_daily = (
    procedureevents
    .filter(
        F.col("stay_id").isNotNull()
        & F.col("starttime").isNotNull()
    )
    .withColumn(
        "clinical_date",
        F.to_date("starttime")
    )
    .groupBy(
        "stay_id",
        "clinical_date"
    )
    .agg(
        F.count("*").alias("procedure_count"),
        F.countDistinct("itemid").alias("distinct_procedure_types")
    )
)

display(
    procedure_daily.agg(
        F.count("*").alias("procedure_stay_days"),
        F.sum("procedure_count").alias("procedures")
    )
)

In [0]:
# Assemble ICU daily fact

fact_icu_daily_monitoring = (
    icu_daily_spine

    .join(
        vital_daily,
        ["stay_id", "clinical_date"],
        "left"
    )
    .join(
        lab_daily,
        ["stay_id", "clinical_date"],
        "left"
    )
    .join(
        input_daily,
        ["stay_id", "clinical_date"],
        "left"
    )
    .join(
        output_daily,
        ["stay_id", "clinical_date"],
        "left"
    )
    .join(
        procedure_daily,
        ["stay_id", "clinical_date"],
        "left"
    )

    # Shared dimension keys
    .join(
        dim_patient.select(
            "subject_id",
            "patient_key"
        ),
        "subject_id",
        "left"
    )

    .withColumn(
        "date_key",
        F.date_format(
            "clinical_date",
            "yyyyMMdd"
        ).cast("int")
    )

    # Retain whether fluids were actually recorded
    .withColumn(
        "fluid_balance_complete",
        F.col("total_input_ml").isNotNull()
        & F.col("total_output_ml").isNotNull()
    )

    .withColumn(
        "net_fluid_balance_ml",
        F.when(
            F.col("fluid_balance_complete"),
            F.col("total_input_ml")
            - F.col("total_output_ml")
        ).otherwise(
            F.lit(None).cast("double")
        )
    )

    # Missing activity counts mean zero recorded retained events
    .fillna({
        "vital_observation_count": 0,
        "distinct_vital_measurements": 0,
        "vital_warning_count": 0,
        "lab_result_count": 0,
        "distinct_lab_tests": 0,
        "abnormal_lab_count": 0,
        "input_event_count": 0,
        "output_event_count": 0,
        "procedure_count": 0,
        "distinct_procedure_types": 0
    })

    # Analytical flags
    .withColumn(
        "abnormal_lab_flag",
        F.col("abnormal_lab_count") > 0
    )
    .withColumn(
        "high_vital_activity_flag",
        F.col("vital_observation_count") >= 104
    )
    .withColumn(
        "high_lab_activity_flag",
        F.col("lab_result_count") >= 17
    )
    .withColumn(
        "high_procedure_activity_flag",
        F.col("procedure_count") >= 8
    )
    .withColumn(
        "high_monitoring_activity_flag",
        F.col("high_vital_activity_flag")
        | F.col("high_lab_activity_flag")
    )
    .withColumn(
        "large_fluid_shift_flag",
        F.when(
            F.col("net_fluid_balance_ml").isNotNull(),
            F.abs(F.col("net_fluid_balance_ml")) >= 4465.29
        ).otherwise(False)
    )
)

In [0]:
# Add ICU careunit dimension key
# The main ICU careunit is the stay's first careunit.

fact_icu_daily_monitoring = (
    fact_icu_daily_monitoring

    .join(
        dim_careunit
        .select(
            F.col("careunit_name").alias("first_careunit"),
            F.col("careunit_key").alias("careunit_key")
        ),
        "first_careunit",
        "left"
    )

    .select(
        "subject_id",
        "hadm_id",
        "stay_id",
        "clinical_date",

        "patient_key",
        "date_key",
        "careunit_key",

        "icu_day_number",
        "icu_los_days",
        "icu_stay_complete_flag",
        "los_availability_flag",

        "vital_observation_count",
        "distinct_vital_measurements",
        "vital_warning_count",

        "heart_rate_avg",
        "heart_rate_min",
        "heart_rate_max",

        "respiratory_rate_avg",
        "respiratory_rate_min",
        "respiratory_rate_max",

        "spo2_avg",
        "spo2_min",
        "spo2_max",

        "temperature_c_avg",
        "temperature_c_min",
        "temperature_c_max",

        "lab_result_count",
        "distinct_lab_tests",
        "abnormal_lab_count",
        "abnormal_lab_flag",

        "creatinine_avg",
        "hemoglobin_avg",
        "wbc_avg",
        "sodium_avg",
        "potassium_avg",
        "lactate_avg",

        "input_event_count",
        "total_input_ml",

        "output_event_count",
        "total_output_ml",

        "fluid_balance_complete",
        "net_fluid_balance_ml",

        "procedure_count",
        "distinct_procedure_types",

        "high_vital_activity_flag",
        "high_lab_activity_flag",
        "high_procedure_activity_flag",
        "high_monitoring_activity_flag",
        "large_fluid_shift_flag"
    )
)

In [0]:
# Validate ICU daily fact before publishing

icu_fact_validation = (
    fact_icu_daily_monitoring
    .agg(
        F.count("*").alias("rows"),

        F.countDistinct(
            F.struct(
                "stay_id",
                "clinical_date"
            )
        ).alias("unique_stay_days"),

        F.countDistinct("stay_id").alias("unique_stays"),

        F.sum(
            F.when(
                F.col("vital_observation_count") > 0,
                1
            ).otherwise(0)
        ).alias("vital_days"),

        F.sum(
            F.when(
                F.col("lab_result_count") > 0,
                1
            ).otherwise(0)
        ).alias("lab_days"),

        F.sum(
            F.when(
                F.col("procedure_count") > 0,
                1
            ).otherwise(0)
        ).alias("procedure_days"),

        F.sum(
            F.when(
                F.col("total_input_ml").isNotNull(),
                1
            ).otherwise(0)
        ).alias("input_days"),

        F.sum(
            F.when(
                F.col("total_output_ml").isNotNull(),
                1
            ).otherwise(0)
        ).alias("output_days"),

        F.sum(
            F.when(
                F.col("fluid_balance_complete"),
                1
            ).otherwise(0)
        ).alias("complete_fluid_days"),

        F.sum(
            F.when(
                F.col("patient_key").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_patient_keys"),

        F.sum(
            F.when(
                F.col("date_key").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_date_keys"),

        F.sum(
            F.when(
                F.col("careunit_key").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_careunit_keys")
    )
)

display(icu_fact_validation)

In [0]:
# Publish ICU daily fact to Gold

output_df = fact_icu_daily_monitoring.withColumn(
    "_gold_timestamp",
    F.current_timestamp()
)

(
    output_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{GOLD_SCHEMA}.fact_icu_daily_monitoring"
    )
)

print(
    f"Published: "
    f"{GOLD_SCHEMA}.fact_icu_daily_monitoring"
)

In [0]:
# Final persisted validation

persisted_icu_fact = spark.table(
    f"{GOLD_SCHEMA}.fact_icu_daily_monitoring"
)

validation_rows = [
    {
        "table": "fact_icu_daily_monitoring",
        "rows": persisted_icu_fact.count(),
        "unique_grain_rows": (
            persisted_icu_fact
            .select(
                "stay_id",
                "clinical_date"
            )
            .dropDuplicates()
            .count()
        )
    }
]

validation_df = (
    spark.createDataFrame(validation_rows)
    .withColumn(
        "status",
        F.when(
            F.col("rows") == F.col("unique_grain_rows"),
            "PASS"
        ).otherwise("CHECK")
    )
)

display(validation_df)